<a href="https://colab.research.google.com/github/Ekimxii/fitness_pose_tracker/blob/main/form_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("muhannadtuameh/exercise-recognition")
print(path)

100%|██████████| 1.07M/1.07M [00:00<00:00, 80.5MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/muhannadtuameh/exercise-recognition/versions/5


In [2]:
import pandas as pd

names = ["labels", "landmarks", "angles", "3d_distances", "xyz_distances"]
dfs = {n: pd.read_csv(f"{path}/{n}.csv") for n in names}
for n, df in dfs.items():
    print(f"{n:14} {df.shape}  {list(df.columns[:4])}")

# All five files share one ID column; find it instead of guessing its name
shared = set.intersection(*(set(df.columns) for df in dfs.values()))
assert len(shared) == 1, f"Expected one shared ID column, got {shared}"
ID = shared.pop()
LABEL = [c for c in dfs["labels"].columns if c != ID][0]

# Join into one table, one row per frame; validate= stops with an error if an ID repeats
data = dfs["labels"]
for n in names[1:]:
    data = data.merge(dfs[n], on=ID, validate="one_to_one", suffixes=("", f"_{n}"))
print("ID:", ID, "| label:", LABEL, "| merged:", data.shape)

labels         (1372, 2)  ['pose_id', 'pose']
landmarks      (1372, 100)  ['pose_id', 'x_nose', 'y_nose', 'z_nose']
angles         (1372, 8)  ['pose_id', 'right_elbow_right_shoulder_right_hip', 'left_elbow_left_shoulder_left_hip', 'right_knee_mid_hip_left_knee']
3d_distances   (1372, 17)  ['pose_id', 'left_shoulder_left_wrist', 'right_shoulder_right_wrist', 'left_hip_left_ankle']
xyz_distances  (1372, 49)  ['pose_id', 'x_left_shoulder_left_wrist', 'y_left_shoulder_left_wrist', 'z_left_shoulder_left_wrist']
ID: pose_id | label: pose | merged: (1372, 172)


In [3]:
print("missing values:", data.isna().sum().sum())
print(data[LABEL].value_counts())   # expect 10 classes: 5 exercises × up/down

# Split each label into exercise + position
data["position"] = data[LABEL].str.extract(r"(up|down)$", expand=False)
data["exercise"] = data[LABEL].str.replace(r"_?(up|down)$", "", regex=True)
pd.crosstab(data["exercise"], data["position"])

missing values: 0
pose
jumping_jacks_down    189
jumping_jacks_up      181
pullups_down          154
pushups_up            144
squats_up             139
pullups_up            135
squats_down           127
pushups_down          102
situp_down            102
situp_up               99
Name: count, dtype: int64


position,down,up
exercise,,
jumping_jacks,189,181
pullups,154,135
pushups,102,144
situp,102,99
squats,127,139
